## Microsoft Agent Framework (MAF): Ramp-Up Training Materials

To get the latest version of MAF Python package, use:

``` bash
pip install --upgrade agent-framework
pip install --upgrade azure-ai-projects
```

## 📒 Notebook 4: Memory

### 🪜 Step 1: Configure Environment

In [1]:
# Import required packages
import os
import json
import asyncio
from agent_framework import AgentSession
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import DefaultAzureCredential

In [2]:
# Set environment variables
PROJECT_ENDPOINT = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_GPT_MODEL")

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    print("WARNING: Environment variables not set properly!")
else:
    print("Environment variables set successfully!")

Environment variables set successfully!


### 🪜 Step 2: Create AI Agent

In [3]:
# Define Foundry client
client = FoundryChatClient(
    project_endpoint = PROJECT_ENDPOINT,
    model = MODEL_DEPLOYMENT,
    credential = DefaultAzureCredential()
)

# Create AI agent
agent = client.as_agent(
    name = "hotel-concierge-agent",
    instructions = """
    You are a friendly and professional hotel concierge. 
    Always remember the guest's name and assign them a room number during their check in.
    """
)

print(f"Agent created: {agent.name}")

Agent created: hotel-concierge-agent


### 🪜 Step 3: Run Agent and Serialise Session

In [4]:
# Create session for the guest conversation
guest_session = agent.create_session()

print(f"New guest session created")

New guest session created


In [5]:
# Simulate guest check-in
PROMPT1 = "Hello, my name is Alex Reed and I am checking in."
print(f"User (Check-in):\n{PROMPT1}\n")

response = await agent.run(PROMPT1, session=guest_session)
print(f"Agent:\n{response.text}\n")

User (Check-in):
Hello, my name is Alex Reed and I am checking in.

Agent:
Welcome, Alex Reed! It’s a pleasure to have you with us. I am assigning you room 402. If you need any assistance or information during your stay, please don't hesitate to ask. Enjoy your stay!



In [6]:
# Serialise the session state
serialised_session_data = guest_session.to_dict()

# Save thread state to a local JSON file
FILE_NAME = "concierge_session_memory.json"
with open(FILE_NAME, "w") as f:
    json.dump(serialised_session_data, f, indent=4)
    
print(f"Session data successfully serialised and saved to {FILE_NAME}")

Session data successfully serialised and saved to concierge_session_memory.json


### 🪜 Step 4: Application Restart

In [7]:
# Close Foundry client and delete all objects from memory
del agent
del client
del guest_session

print("Foundry client, agent and session objects cleared to simulate memory loss")

Foundry client, agent and session objects cleared to simulate memory loss


### 🪜 Step 5: Re-initialise Objects and Deserialise Session

In [ ]:
# Re-create Foundry client and agent
client_new = FoundryChatClient(
    project_endpoint = PROJECT_ENDPOINT,
    model = MODEL_DEPLOYMENT,
    credential = DefaultAzureCredential()
)

agent_new = client_new.as_agent(
    name = "hotel-concierge-agent",
    instructions = """
    You are a friendly and professional hotel concierge. 
    Always remember the guest's name and assign them a room number during their check in.
    """
)

print(f"New Foundry client and agent objects created after simulated memory loss")

New Foundry client and agent objects created after simulated memory loss


In [9]:
# Load and deserialise session from JSON file
with open(FILE_NAME, "r") as f:
    session_data_reloaded = json.load(f)
    
restored_session = AgentSession.from_dict(session_data_reloaded)
print(f"Session successfully deserialised. Type: {type(restored_session)}")

Session successfully deserialised. Type: <class 'agent_framework._sessions.AgentSession'>


In [10]:
# Continue conversation using the restored thread
PROMPT2 = "I would like to order room service: a club sandwich and a pot of tea."
print(f"User (Room Service):\n{PROMPT2}\n")

response_final = await agent_new.run(PROMPT2, session=restored_session)
print(f"Agent:\n{response_final.text}")

User (Room Service):
I would like to order room service: a club sandwich and a pot of tea.

Agent:
Certainly, Alex! I have placed an order for a club sandwich and a pot of tea to be delivered to room 402. Is there anything else you would like to add or any preferences for your tea?
